Comparative Evaluation：Diffusion models vs Autoregressive models


1. Fine tune autoregressive model Llama 3.1 (baseline)
2. Test on test set
3. Metrics
4. LlaDa, LAD
5. Compare

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
pip install rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d839986b3c5f36d68e9f705695f8007725ef19a823180de555626d61b78a2258
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [4]:
import torch
import gc

# 清理显存
torch.cuda.empty_cache()
gc.collect()

# 查看当前显存使用情况
print(f"已分配: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
print(f"已缓存: {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")

# 如果有其他进程占用，让Python强制清理
torch.cuda.synchronize()

已分配: 0.00 GB
已缓存: 0.00 GB


In [5]:
#deep learning
import torch

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from huggingface_hub import login
from getpass import getpass

import pandas as pd
import numpy as np

#Metrics
from rouge_score import rouge_scorer
import bert_score

import nltk
from collections import Counter
import string
from wordcloud import WordCloud

import matplotlib.pyplot as plt

import os
import zipfile

In [6]:
base_path = os.getcwd()
data_roots = [
    'multiclinsum_gs_train_en',
    'multiclinsum_large-scale_train_en',
    'multiclinsum_test_en'
]

dfs = {}

for folder in data_roots:
    zip_path = os.path.join(base_path, f'{folder}.zip')
    records = []

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Iterate through all files in the zip
        for file in z.namelist():
            if '/fulltext/' in file and file.endswith('.txt'):
                # Read the full text
                full_text = z.read(file).decode('utf-8')

                # Build corresponding summary file path
                summary_file = file.replace('/fulltext/', '/summaries/').replace('.txt', '_sum.txt')
                summary_text = z.read(summary_file).decode('utf-8')

                records.append({'Full_Text': full_text, 'Summary': summary_text})

    dfs[folder] = pd.DataFrame(records)

df_gs = dfs['multiclinsum_gs_train_en']
df_large = dfs['multiclinsum_large-scale_train_en']
df_test = dfs['multiclinsum_test_en']

In [7]:
#Prepare training data
def format_example(row):
    return {"text": f"Summarize this clinical note: {row['Full_Text']}\nSummary: {row['Summary']}"}

train_data = [format_example(row) for _, row in df_gs.iterrows()]
dataset = Dataset.from_list(train_data)

In [8]:
hf_token = getpass("Enter your Hugging Face token: ")

model_name = "meta-llama/Meta-Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

Enter your Hugging Face token: ··········


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [9]:
#Lora
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)


model = get_peft_model(model, lora_config)

In [10]:
#tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/592 [00:00<?, ? examples/s]

In [11]:
# Training parameters
training_args = TrainingArguments(
    output_dir="./llama-clinical-summary",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=500,
    bf16=True,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

Step,Training Loss
10,1.930466
20,1.870416
30,1.857782
40,1.813190
50,1.854955
60,1.874238
70,1.867859
80,1.829627
90,1.798855
100,1.788738


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69ea0373-002c7aeb4ed83f3f172e919a;09046a53-d166-42f9-91e9-3f5c875e80d6)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=222, training_loss=1.7855004448074479, metrics={'train_runtime': 1023.6178, 'train_samples_per_second': 1.735, 'train_steps_per_second': 0.217, 'total_flos': 4.098309420692275e+16, 'train_loss': 1.7855004448074479, 'epoch': 3.0})

In [12]:
model.save_pretrained("./llama-clinical-summary-lora")
tokenizer.save_pretrained("./llama-clinical-summary-lora")

/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69ea0378-6a5da01c7fe8deed1460f374;c5314128-7738-4ac2-b465-051c9f315c24)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B - will assume that the vocabulary was not modified.
  warnings.warn(


('./llama-clinical-summary-lora/tokenizer_config.json',
 './llama-clinical-summary-lora/tokenizer.json')